In [ ]:
#Import required libraries for file handling, image I/O, and resampling techniques
import os
import glob
import nibabel as nib
import numpy as np
import SimpleITK as sitk

def resample_nibabel_image(img, output_spacing=(2.0, 2.0, 2.0), output_size=(128, 128, 128)):
    #Convert the image data to a NumPy array and extract the original voxel spacing
    img_data = img.get_fdata()
    orig_spacing = [float(x) for x in img.header.get_zooms()]

    #Convert NumPy image to SimpleITK format and assign its original spacing
    sitk_img = sitk.GetImageFromArray(img_data)
    sitk_img.SetSpacing(orig_spacing)

    #Resample the image to the specified spacing and size
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(output_spacing)
    resampler.SetSize(output_size)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampled_sitk = resampler.Execute(sitk_img)
    
    #Convert back to NumPy and create a new NIfTI image with updated affine
    resampled_data = sitk.GetArrayFromImage(resampled_sitk)
    new_affine = np.diag(list(output_spacing) + [1])
    resampled_img = nib.Nifti1Image(resampled_data, affine=new_affine)
    
    return resampled_img

#Define input dataset directories and corresponding output folders
dataset_directs = {
    'mri_ad': 'DATA/MRI_AD',
    'mri_cn': 'DATA/MRI_CN',
    'pet_ad': 'DATA/PET_NIFTI_AD',
    'pet_cn': 'DATA/PET_NIFTI_CN'
}

output_directs = {
    'mri_ad': 'mri_ad_vsschanges',
    'mri_cn': 'mri_cv_vsschanges',
    'pet_ad': 'pet_ad_vsschanges',
    'pet_cn': 'pet_cn_vsschanges'
}

#Create the output directories if they do not already exist
for folder in output_directs.values():
    os.makedirs(folder, exist_ok=True)

In [ ]:
#Locate all NIfTI files within the specified dataset directories
dataset_files = {}
for key, folder in dataset_directs.items():
    files = glob.glob(os.path.join(folder, "**", "*.nii"), recursive=True)
    dataset_files[key] = files
    print(f"Found {len(files)} files in {folder}")


In [ ]:
#Loop through each dataset and apply resampling to every image
for key, file_list in dataset_files.items():
    for i, file_path in enumerate(file_list, start=1):
        img = nib.load(file_path)
        res_img = resample_nibabel_image(img, output_spacing=(2.0, 2.0, 2.0), output_size=(128, 128, 128))
        out_filename = f"resampled_{key}_{i}.nii"
        out_path = os.path.join(output_directs[key], out_filename)
        nib.save(res_img, out_path)
        print(f"Saved {key}: {out_path}")


In [ ]:
#Function to check whether resampled images match the expected shape and spacing
def check_images_format(output_dir, expected_shape=(128, 128, 128), expected_spacing=(2.0, 2.0, 2.0)):
    files = glob.glob(os.path.join(output_dir, "*.nii"))
    print(f"\nChecking {len(files)} files in {output_dir}:")
    for f in files:
        img = nib.load(f)
        data = img.get_fdata()
        header = img.header
        spacing = header.get_zooms()[:3]
        print(f"File: {os.path.basename(f)}")
        print("Data shape:", data.shape)
        print("Spacing:", spacing)
        if data.shape != expected_shape:
            print("ERROR: Unexpected data shape!")
        if spacing != expected_spacing:
            print("ERROR: Unexpected spacing!")
        print("----------------------------------------------")

#Check all output folders to confirm that resampled files meet the intended format
output_directs = {
    'mri_ad': 'mri_ad_vsschanges',
    'mri_cn': 'mri_cv_vsschanges',  
    'pet_ad': 'pet_ad_vsschanges',
    'pet_cn': 'pet_cn_vsschanges'
}

for folder in output_directs.values():
    check_images_format(folder)